# Phase 1 — Lesson 09: Information Theory
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 09: Information Theory
**Type:** Learn | **Languages:** Python | **Time:** ~60 minutes
**Prerequisites:** Phase 1, Lesson 06 (Probability)



# What You'll Learn

---

## Learning Objectives

- Compute entropy, cross-entropy, and KL divergence from scratch and explain their relationship
- Derive why minimizing cross-entropy loss is equivalent to maximizing log-likelihood
- Calculate mutual information between features and a target to rank feature importance
- Explain perplexity as the effective vocabulary size a language model chooses from


# The Problem

---

You call `CrossEntropyLoss()` in every classification model you train. You see "perplexity" in every language model paper. These are not disconnected concepts. They are all the same idea wearing different hats.

Information theory gives you the language to reason about uncertainty, compression, and prediction.


# Information Content (Surprise)

---

When something unlikely happens, it carries more information. A coin landing heads? Not surprising. A lottery win? Very surprising.

```
I(x) = -log(p(x))
```

| Event | Probability | Surprise (bits) |
|-------|-------------|-----------------|
| Fair coin heads | 0.5 | 1.0 |
| Rolling a 6 | 0.167 | 2.58 |
| 1-in-1000 event | 0.001 | 9.97 |
| Certain event | 1.0 | 0.0 |
```


In [ ]:
import mathdef information_content(p, base=2):    if p <= 0 or p > 1:        return float('inf') if p <= 0 else 0.0    return -math.log(p) / math.log(base)for p, name in [(0.5, 'fair coin heads'), (0.167, 'rolling a 6'), (0.001, '1-in-1000'), (1.0, 'certain')]:    print(f"{name:20s} p={p:6.3f}  surprise={information_content(p):.2f} bits")

# Entropy: Average Surprise

---

Entropy is the expected surprise across all possible outcomes:

```
H(X) = -sum( p(x) * log(p(x)) )
```

A fair coin has maximum entropy for a binary variable: 1 bit. A biased coin (99% heads) has low entropy: 0.08 bits.


In [ ]:
import mathdef entropy(probs, base=2):    return sum(-p * math.log(p, base) for p in probs if p > 0)print(f"Fair coin:    {entropy([0.5, 0.5]):.4f} bits")print(f"Biased coin:  {entropy([0.99, 0.01]):.4f} bits")print(f"Fair die:     {entropy([1/6]*6):.4f} bits")

# Cross-Entropy and KL Divergence

---

**Cross-entropy:** average surprise when using Q to encode events from P:
```
H(P, Q) = -sum( p(x) * log(q(x)) )
```

**KL divergence:** extra bits wasted when Q approximates P:
```
KL(P || Q) = H(P, Q) - H(P)
```

Minimizing cross-entropy = minimizing KL divergence (entropy of P is constant).


In [ ]:
import mathdef cross_entropy(p, q, base=2):    return sum(-p * math.log(q, base) for p, q in zip(p, q) if p > 0 and q > 0)def kl_divergence(p, q, base=2):    return sum(p * math.log(p / q, base) for p, q in zip(p, q) if p > 0 and q > 0)true = [0.7, 0.2, 0.1]good = [0.6, 0.25, 0.15]bad = [0.1, 0.1, 0.8]H = entropy(true)CE_g = cross_entropy(true, good)CE_b = cross_entropy(true, bad)KL_g = kl_divergence(true, good)KL_b = kl_divergence(true, bad)print(f"Entropy of true:   {H:.4f}")print(f"CE (good model):   {CE_g:.4f}")print(f"CE (bad model):    {CE_b:.4f}")print(f"KL divergence good:{KL_g:.4f}")print(f"KL divergence bad: {KL_b:.4f}")

In [ ]:
import matplotlib.pyplot as pltimport numpy as np# Show CE vs KL relationshiptrue = [0.7, 0.2, 0.1]q2_values = np.linspace(0.05, 0.95, 50)ces = []; kls = []for q2 in q2_values:    q = [q2, (1-q2)*0.8, (1-q2)*0.2]    ces.append(cross_entropy(true, q))    kls.append(kl_divergence(true, q))fig, ax = plt.subplots(figsize=(7, 4))ax.plot(q2_values, ces, 'b-', lw=2, label='Cross-entropy')ax.plot(q2_values, kls, 'r--', lw=2, label='KL divergence')ax.axvline(0.7, color='g', linestyle=':', label='true p=0.7 (minimum)')ax.set_xlabel('q_2 (model prob for class 0)')ax.set_ylabel('Value (bits)')ax.set_title('Cross-Entropy = Entropy + KL Divergence', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Mutual Information

---

Mutual information measures how much knowing one variable tells you about another:

```
I(X; Y) = H(X) - H(X|Y)
```

If X and Y are independent, MI = 0. If perfectly correlated, MI = H(X).

In feature selection: high MI between a feature and the target means the feature is useful.


In [ ]:
import mathdef mutual_information(joint_probs, base=2):    rows = len(joint_probs); cols = len(joint_probs[0])    mx = [sum(joint_probs[i][j] for j in range(cols)) for i in range(rows)]    my = [sum(joint_probs[i][j] for i in range(rows)) for j in range(cols)]    mi = 0.0    for i in range(rows):        for j in range(cols):            pxy = joint_probs[i][j]            if pxy > 0:                mi += pxy * math.log(pxy / (mx[i] * my[j]), base)    return miindependent = [[0.25, 0.25], [0.25, 0.25]]dependent = [[0.45, 0.05], [0.05, 0.45]]print(f"MI (independent): {mutual_information(independent):.4f} bits")print(f"MI (dependent):   {mutual_information(dependent):.4f} bits")

In [ ]:
import matplotlib.pyplot as pltimport numpy as np# MI heatmap for joint distributionsfig, axes = plt.subplots(1, 2, figsize=(10, 4))for ax, J, title in [(axes[0], independent, 'Independent (MI=0)'), (axes[1], dependent, 'Dependent (MI>0)')]:    im = ax.imshow(J, cmap='Blues', vmin=0, vmax=0.5)    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])    ax.set_xticklabels(['Y=0', 'Y=1']); ax.set_yticklabels(['X=0', 'X=1'])    for i in range(2):        for j in range(2):            ax.text(j, i, f'{J[i][j]:.2f}', ha='center', va='center', fontweight='bold')    ax.set_title(title, fontweight='bold')    plt.colorbar(im, ax=ax)plt.tight_layout(); plt.show()

# Perplexity

---

Perplexity is the exponential of cross-entropy. It tells you the effective number of equally likely choices the model is uncertain between.

```
Perplexity = 2^H(P,Q)   (bits)  or  e^H(P,Q)   (nats)
```

A language model with perplexity 50 is, on average, as confused as if it had to pick uniformly from 50 possible next tokens. Lower is better.


In [ ]:
import mathlogits = [2.0, 1.0, 0.1]true_class = 0def softmax(logits):    m = max(logits)    exps = [math.exp(z - m) for z in logits]    t = sum(exps)    return [e / t for e in exps]probs = softmax(logits)loss = -math.log(probs[true_class])perp = math.exp(loss)print(f"Logits: {logits}")print(f"Softmax: {[f'{p:.4f}' for p in probs]}")print(f"Cross-entropy: {loss:.4f} nats")print(f"Perplexity: {perp:.2f}")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** When is entropy maximized?
> **Explanation:** When the distribution is uniform — maximum uncertainty.

**Q2:** Cross-entropy minimized when?
> **Explanation:** When predicted = true (KL = 0).

**Q3:** What does perplexity tell you?
> **Explanation:** The effective vocabulary size the model is choosing from at each step.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Compute entropy of the English alphabet (uniform, 26 letters) vs actual letter frequencies.

**Exercise 2:** Show KL divergence is not symmetric: pick two distributions and compute both directions.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Entropy | Randomness | Average surprise across all outcomes — irreducible uncertainty |
| Cross-entropy | The loss function | Average surprise using Q to encode P — used as classification loss |
| KL divergence | Distance between distributions | Extra bits wasted when Q approximates P — always ≥ 0 |
| Mutual information | How related are X and Y | Reduction in uncertainty about X from knowing Y |
| Perplexity | How confused the model is | Exponential of cross-entropy — effective vocabulary size |


# Further Reading

---

- [Shannon 1948: A Mathematical Theory of Communication](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf)
- [Visual Information Theory (Chris Olah)](https://colah.github.io/posts/2015-09-Visual-Information/)


# Next Steps

---

When you're ready, continue to Lesson 11: Singular Value Decomposition.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
